<a href="https://colab.research.google.com/github/LegalIntermediaSL/Nautica/blob/main/simulaciones/16_curva_polar_velero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulación 16: Curva Polar de Velocidad y VMG

La **curva polar** de un velero representa, para cada ángulo de viento aparente/real y cada intensidad de viento, la velocidad que puede alcanzar el barco. Es la herramienta central de cualquier software de ruta meteorológica (routing): a partir de ella se decide el rumbo óptimo para ceñir o para ir largo.

## VMG (Velocity Made Good)

No siempre el rumbo con más velocidad de casco es el que más rápido te acerca al punto de destino que está *justo contra el viento* o *justo a favor*. El **VMG** es la componente de la velocidad del barco proyectada sobre la dirección del viento:

$$ VMG = V_{barco} \times \cos(\theta) $$

donde $\theta$ es el ángulo entre el rumbo del barco y la dirección del viento (True Wind Angle, TWA).
*   **Al ceñir** (TWA pequeño, viento por la proa): interesa el ángulo que **maximiza** $V_{barco} \cdot \cos(\theta)$ — normalmente no es el ángulo más cerrado posible, porque muy cerca del viento el barco apenas avanza.
*   **Al largo/empopada** (TWA grande, viento por la popa): interesa el ángulo que **maximiza** $V_{barco} \cdot \cos(180° - \theta)$ — a veces conviene "abrir" el rumbo respecto al destino directo para ganar mucha más velocidad (surfear las olas, evitar la pérdida de presión en las velas dead-downwind).

Esta simulación usa un **modelo simplificado** de rendimiento vela-casco (no viene de un VPP -Velocity Prediction Program- real ni de polares de fabricante), solo para ilustrar la forma característica de la curva y el cálculo del VMG óptimo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def velocidad_barco(angulo_grados, viento_real_nudos):
    """
    Modelo didáctico simplificado de la velocidad de un crucero-velero
    en función del ángulo de viento real (0-180 grados) y su intensidad.
    No es un VPP real: solo reproduce la forma típica de una polar.
    """
    angulo_grados = np.asarray(angulo_grados, dtype=float)
    velocidad = np.zeros_like(angulo_grados)

    # Zona de no-ceñida (no se puede navegar a menos de ~30-35º del viento)
    zona_navegable = angulo_grados >= 32

    # Forma tipo "pétalo": crece desde el límite de ceñida, máximo hacia el
    # través/largo (100-130º) y decae ligeramente en la empopada pura.
    factor_angulo = np.sin(np.radians((angulo_grados - 20) * 1.1))
    factor_angulo = np.clip(factor_angulo, 0, None)

    # Rendimiento con la intensidad de viento: crece con la raíz del viento
    # (limitado por la resistencia del casco: rendimientos decrecientes)
    factor_viento = 0.62 * np.sqrt(viento_real_nudos)

    velocidad_calc = factor_angulo * factor_viento
    velocidad_maxima_casco = 1.34 * np.sqrt(9.0)  # velocidad de casco aprox. (LWL=9m)

    velocidad = np.where(zona_navegable, np.minimum(velocidad_calc, velocidad_maxima_casco), 0.0)
    return velocidad


# --- ÁNGULOS Y VIENTOS A EVALUAR ---
angulos = np.linspace(30, 180, 151)          # grados, de ceñida a empopada
vientos_reales = [10, 20, 30]                # nudos

resultados = {}
for tws in vientos_reales:
    v_barco = velocidad_barco(angulos, tws)
    vmg_cenida = v_barco * np.cos(np.radians(angulos))              # positivo al ceñir
    vmg_largo = v_barco * np.cos(np.radians(180 - angulos))         # positivo al largo

    mascara_cenida = angulos <= 90
    mascara_largo = angulos >= 90

    idx_c = np.argmax(vmg_cenida[mascara_cenida])
    idx_l = np.argmax(vmg_largo[mascara_largo])

    ang_optimo_cenida = angulos[mascara_cenida][idx_c]
    vmg_optimo_cenida = vmg_cenida[mascara_cenida][idx_c]
    v_en_optimo_cenida = v_barco[mascara_cenida][idx_c]

    ang_optimo_largo = angulos[mascara_largo][idx_l]
    vmg_optimo_largo = vmg_largo[mascara_largo][idx_l]
    v_en_optimo_largo = v_barco[mascara_largo][idx_l]

    resultados[tws] = {
        "v_barco": v_barco,
        "cenida": (ang_optimo_cenida, v_en_optimo_cenida, vmg_optimo_cenida),
        "largo": (ang_optimo_largo, v_en_optimo_largo, vmg_optimo_largo),
    }

    print(f"--- Viento Real: {tws} nudos ---")
    print(f"  Mejor ángulo al CEÑIR:  {ang_optimo_cenida:5.0f}º -> V={v_en_optimo_cenida:4.1f} kn, VMG={vmg_optimo_cenida:4.1f} kn")
    print(f"  Mejor ángulo al LARGO:  {ang_optimo_largo:5.0f}º -> V={v_en_optimo_largo:4.1f} kn, VMG={vmg_optimo_largo:4.1f} kn")

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="polar")
ax.set_theta_zero_location("N")   # 0º viento por la proa, arriba
ax.set_theta_direction(-1)        # sentido horario, como una brújula

colores = {10: "tab:blue", 20: "tab:green", 30: "tab:red"}

for tws in vientos_reales:
    v_barco = resultados[tws]["v_barco"]
    ax.plot(np.radians(angulos), v_barco, label=f"{tws} nudos de viento real", color=colores[tws], linewidth=2)

    # Marcar el punto óptimo de ceñida y de largo
    ang_c, v_c, _ = resultados[tws]["cenida"]
    ang_l, v_l, _ = resultados[tws]["largo"]
    ax.plot(np.radians(ang_c), v_c, "o", color=colores[tws])
    ax.plot(np.radians(ang_l), v_l, "s", color=colores[tws])

ax.set_title("Curva Polar de Velocidad (crucero tipo)\n○ = óptimo de ceñida   □ = óptimo de largo", pad=30)
ax.set_thetamin(0)
ax.set_thetamax(180)
ax.legend(loc="lower left", bbox_to_anchor=(-0.15, -0.05))
plt.tight_layout()
plt.show()

## Conclusión

La curva polar muestra por qué un velero nunca navega directamente hacia el punto de donde sopla el viento ni, casi nunca, exactamente en la dirección contraria: el **VMG óptimo al ceñir** suele estar entre 40º y 50º del viento real, y el **VMG óptimo al largo** casi nunca es la empopada pura (180º), sino un ángulo algo más cerrado donde el barco gana mucha más velocidad de casco a cambio de perder poco ángulo directo. Cuanto más viento real disponible, mayor es la velocidad alcanzable en todos los ángulos (las curvas de 20 y 30 nudos envuelven a la de 10 nudos). En la práctica, estas curvas se obtienen mediante ensayos reales o programas VPP específicos del diseño de cada barco, y son la base de cualquier sistema de routing meteorológico.